# Face Recognition & Quality Estimation with ArcFace

**Project Overview:**
This notebook implements an Identity Verification system using Metric Learning.
Instead of classifying images into fixed categories, which doesn't scale to new people, we learn a continuous vector space where faces of the same person are close together.

**Engineering Approach:**
1.  **ArcFace Loss:** We use Additive Angular Margin Loss to enforce tighter intra-class variance and wider inter-class separation than standard Softmax.
2.  **Embedding Analysis:** We visualize the distribution of Cosine Similarities to determine the optimal decision threshold.
3.  **Trash Detection:** We test the hypothesis that the L2-norm of the unnormalized embedding correlates with image quality.

**Structure:**
*   `src.backbone`: ResNet18 modified to return `(embedding, norm)`.
*   `src.metric_loss`: Custom ArcFace implementation.
*   `src.dataset`: CelebA loader handling identity labels.

In [ ]:
import sys
import os
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader

sys.path.append('../src')

from dataset import CelebAIdentityDataset
from backbone import FaceResNet
from metric_loss import ArcFaceLoss
from utils import seed_everything, get_cosine_similarity

CONFIG = {
    "DATA_DIR": "../data/celeba",
    "WEIGHTS_DIR": "../weights",
    "BATCH_SIZE": 128,
    "LR_BACKBONE": 1e-4,
    "LR_HEAD": 1e-3,
    "EMBEDDING_SIZE": 128,
    "EPOCHS": 10,
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "SEED": 42
}

seed_everything(CONFIG['SEED'])
os.makedirs(CONFIG['WEIGHTS_DIR'], exist_ok=True)
print(f"Running on: {CONFIG['DEVICE']}")

## 1. Data Preparation

We'll load the CelebA dataset. Unlike the VAE project, here we specifically need the Identity Labels.
*   **Preprocessing:** Images are resized to 112x112 of standard resolution for ArcFace/InsightFace pipelines.
*   **Normalization:** We use `[-1, 1]` normalization which is standard for Face Recognition tasks.

In [ ]:
train_ds = CelebAIdentityDataset(CONFIG['DATA_DIR'], split='train')
val_ds = CelebAIdentityDataset(CONFIG['DATA_DIR'], split='val')
test_ds = CelebAIdentityDataset(CONFIG['DATA_DIR'], split='test')

train_loader = DataLoader(train_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=4)

num_classes = len(set(train_ds.labels))
print(f"Training Images: {len(train_ds)} | Identities: {num_classes}")
print(f"Validation Images: {len(val_ds)}")

## 2. Model Architecture & Loss

We initialize two components:
1.  **Backbone, `FaceResNet`:** Extracts a 128-d vector from the image.
2.  **Loss Head, `ArcFaceLoss`:** Maintains the "Class Centers" and computes the angular margin loss.

In [ ]:
backbone = FaceResNet(embedding_size=CONFIG['EMBEDDING_SIZE']).to(CONFIG['DEVICE'])

metric_loss = ArcFaceLoss(
    in_features=CONFIG['EMBEDDING_SIZE'],
    out_features=num_classes,
    s=30.0,
    m=0.5
).to(CONFIG['DEVICE'])

optimizer = optim.AdamW([
    {'params': backbone.parameters(), 'lr': CONFIG['LR_BACKBONE']},
    {'params': metric_loss.parameters(), 'lr': CONFIG['LR_HEAD']}
], weight_decay=1e-4)

print("Model Initialized.")

## 3. Training Loop

This loop is slightly different from standard classification.
*   **Forward:** Backbone returns `(embedding, norm)`.
*   **Loss:** ArcFace takes the normalized embedding and the label.

In [ ]:
def run_epoch(loader, is_train=True):
    backbone.train() if is_train else backbone.eval()

    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Train" if is_train else "Val", leave=False)

    for images, labels in pbar:
        images, labels = images.to(CONFIG['DEVICE']), labels.to(CONFIG['DEVICE'])

        with torch.set_grad_enabled(is_train):
            embeddings, _ = backbone(images)

            loss = metric_loss(embeddings, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item()
        total += labels.size(0)

        pbar.set_postfix({"Loss": loss.item()})

    return total_loss / len(loader)

In [ ]:
TRAIN_MODEL = True

history = {"train_loss": [], "val_loss": []}

if TRAIN_MODEL:
    print("Starting Training...")
    for epoch in range(CONFIG['EPOCHS']):
        t_loss = run_epoch(train_loader, is_train=True)
        v_loss = run_epoch(val_loader, is_train=False)

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)

        print(f"Epoch {epoch+1}/{CONFIG['EPOCHS']} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f}")

    torch.save(backbone.state_dict(), f"{CONFIG['WEIGHTS_DIR']}/arcface_backbone.pth")
    print("Backbone Saved.")
else:
    if os.path.exists(f"{CONFIG['WEIGHTS_DIR']}/arcface_backbone.pth"):
        backbone.load_state_dict(torch.load(f"{CONFIG['WEIGHTS_DIR']}/arcface_backbone.pth"))
        print("Backbone Weights Loaded.")
    else:
        print("No weights found.")

## 4. Analysis: Cosine Similarity Distribution

In a deployed system, we don't have "classes". We just have two images, and we ask: "Is this the same person?"
We answer this by calculating the Cosine Similarity.

*   **Hypothesis:** Pairs of the Same Identity should have high similarity, close to 1. Pairs of Different Identities should have low similarity, close to 0.
*   **Goal:** Visualize the overlap between these two distributions.

In [ ]:
def get_all_embeddings(loader):
    backbone.eval()
    all_embeddings = []
    all_labels = []
    all_norms = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting"):
            images = images.to(CONFIG['DEVICE'])
            emb, norm = backbone(images)

            all_embeddings.append(emb.cpu())
            all_labels.append(labels)
            all_norms.append(norm.cpu())

    return torch.cat(all_embeddings), torch.cat(all_labels), torch.cat(all_norms)

print("Extracting Test Embeddings...")
test_emb, test_labels, test_norms = get_all_embeddings(test_loader)
print(f"Extracted {len(test_emb)} vectors.")

In [ ]:
num_pairs = 5000
indices = torch.randperm(len(test_emb))[:num_pairs]

same_scores = []
diff_scores = []

print("Computing Pairwise Similarities...")
for i in tqdm(indices):
    query_emb = test_emb[i].unsqueeze(0)
    query_label = test_labels[i]

    sims = get_cosine_similarity(query_emb, test_emb).squeeze()

    is_same = (test_labels == query_label)
    is_diff = ~is_same

    is_same[i] = False

    same_scores.extend(sims[is_same].numpy())

    diff_scores.extend(np.random.choice(sims[is_diff].numpy(), size=min(50, is_diff.sum()), replace=False))

print(f"Same Pairs: {len(same_scores)}")
print(f"Diff Pairs: {len(diff_scores)}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(diff_scores, bins=50, alpha=0.6, color='red', density=True, label='Different Identity')
plt.hist(same_scores, bins=50, alpha=0.6, color='green', density=True, label='Same Identity')

plt.title("Cosine Similarity Distribution (ArcFace)")
plt.xlabel("Cosine Similarity")
plt.ylabel("Density")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. Automated Data Cleaning

**Hypothesis:** The magnitude, L2 Norm, of the embedding before normalization correlates with image quality.
*   **High Norm:** Clear, frontal, high-confidence faces.
*   **Low Norm:** Blurry, side-profile, occluded, or "trash" images.

This allows us to automatically filter bad data from a dataset without manual labeling.

In [ ]:
def denorm_img(tensor):
    return tensor.permute(1, 2, 0) * 0.5 + 0.5

def visualize_quality(norms, dataset, top_k=5):
    sorted_indices = torch.argsort(norms.flatten())

    worst_indices = sorted_indices[:top_k]

    best_indices = sorted_indices[-top_k:]

    plt.figure(figsize=(15, 4))
    for i, idx in enumerate(worst_indices):
        img_tensor, _ = dataset[idx]
        plt.subplot(1, top_k, i+1)
        plt.imshow(denorm_img(img_tensor))
        plt.title(f"Low Norm: {norms[idx].item():.2f}")
        plt.axis('off')
    plt.suptitle("Lowest Embedding Norms (Low Quality / Ambiguous)")
    plt.show()

    plt.figure(figsize=(15, 4))
    for i, idx in enumerate(best_indices):
        img_tensor, _ = dataset[idx]
        plt.subplot(1, top_k, i+1)
        plt.imshow(denorm_img(img_tensor))
        plt.title(f"High Norm: {norms[idx].item():.2f}")
        plt.axis('off')
    plt.suptitle("Highest Embedding Norms (High Quality / Frontal)")
    plt.show()

print("Analyzing Image Quality via Embedding Norms...")
visualize_quality(test_norms, test_ds, top_k=6)